In [70]:
from pymilvus import (
    connections,
    FieldSchema, CollectionSchema, DataType,
    Collection
)
from pymilvus import connections




In [71]:
from pymilvus import utility

# Se a coleção já existir, dropa
if utility.has_collection("image_descriptions"):
    utility.drop_collection("image_descriptions")

# Agora cria do zero
collection = Collection("image_descriptions", schema)


In [77]:
    # se não existir índice, cria
    if not collection.has_index():
        collection.create_index(
            field_name="embedding",
            index_params={
                "metric_type": "COSINE",
                "index_type": "IVF_FLAT",
                "params": {"nlist": 128}
            }
        )

    # carregar coleção
    collection.load()


In [78]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer
import uuid, requests
from PIL import Image, ImageEnhance
import cv2, numpy as np, easyocr
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

# ---- Conexão Milvus ----
connections.disconnect("default")
connections.connect("default", host="localhost", port="19530")

# Definição do schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=36, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # depende do modelo
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="titles", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="texts", dtype=DataType.VARCHAR, max_length=2000),
]
schema = CollectionSchema(fields, description="Armazenamento de imagens processadas")



# Embedding model
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def classify_image(image_path: str) -> str:
    labels = ["Mapa", "Gráfico", "Diagrama", "Tabela", "Outro"]

    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    category = labels[probs.argmax().item()]
    return category


def extract_text_structure(image_path: str) -> dict:
    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    image = image.convert("L")
    image = ImageEnhance.Contrast(image).enhance(2)
    image = ImageEnhance.Sharpness(image).enhance(2)

    image_np = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    reader = easyocr.Reader(["pt"])
    result = reader.readtext(image_np)

    titles, texts = [], []
    for (bbox, text, prob) in result:
        if prob > 0.5:
            if bbox[1][1] < 50:
                titles.append(text.strip())
            else:
                texts.append(text.strip())

    return {"titles": titles, "texts": texts}


def process_and_store(image_url: str):
    try:
        collection.load()
        # Verifica se já existe essa imagem pela URL
        existing = collection.query(
            expr=f'url == "{image_url}"',
            output_fields=["id", "url"],
            limit=1
        )

        if existing:
            print(f"[SKIP] Imagem já existe no Milvus (ID {existing[0]['id']})")
            return
        category = classify_image(image_url)
        extracted = extract_text_structure(image_url)

        description = f"{category}: " + " ".join(extracted["titles"] + extracted["texts"])
        embedding = embedding_model.encode(description).tolist()

        doc_id = str(uuid.uuid4())

        # Inserção no Milvus (cada campo é uma lista, mesmo que só tenha 1 elemento)
        collection.insert([
            [doc_id],
            [embedding],
            [image_url],
            [category],
            [" ".join(extracted["titles"])],
            [" ".join(extracted["texts"])]
        ])

        collection.flush()
        print(f"[OK] Imagem armazenada no Milvus com ID {doc_id}")

    except Exception as e:
        print(f"Erro ao processar/armazenar imagem: {e}")




# Teste
process_and_store("https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID f8cb0b69-9f8a-492a-8e91-6c958e199fb5


In [79]:
def search_images(collection_name, query_text, top_k=3):
    collection = Collection(collection_name)

    # se não existir índice, cria
    if not collection.has_index():
        collection.create_index(
            field_name="embedding",
            index_params={
                "metric_type": "COSINE",
                "index_type": "IVF_FLAT",
                "params": {"nlist": 128}
            }
        )

    # carregar coleção
    collection.load()

    # Gerar embedding do texto
    query_embedding = embedding_model.encode(query_text).tolist()

    # Executar busca
    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["id", "url", "category", "titles", "texts"]
    )

    return results[0]


In [84]:
res = search_images("image_descriptions", "mapa de aguas subterraneas", top_k=3)
for r in res:
    print(f"ID: {r.id}")
    print(f"Score: {r.distance:.4f}")
    print(f"URL: {r.entity.get('url')}")
    print(f"Categoria: {r.entity.get('category')}")
    print(f"Títulos: {r.entity.get('titles')}")
    print(f"Texts: {r.entity.get('texts')}")
    print("-"*40)
    print(f"Distancia: {r.entity.get('distance')}")


ID: 6ab5454f-76c3-4385-9938-3f36c25bcce8
Score: 0.5820
URL: https://smastr16.blob.core.windows.net/2001/sites/261/2024/03/uso-do-solo-800x564.jpg
Categoria: Mapa
Títulos: 
Texts: MAPA DO SOLO URBANO
----------------------------------------
Distancia: 0.58204185962677
ID: 25d954cf-1267-43fa-aef0-e79f8ab3ac2c
Score: 0.5302
URL: https://ufscsustentavel.paginas.ufsc.br/files/2018/11/Volume-Mensal-de-Água-na-UFSC-2017-e-2018.jpg
Categoria: Gráfico
Títulos: Volume Mensal de Água da UFSC 2017 2018 30.000
Texts: 5.000 2017 Z015 Média 2017
----------------------------------------
Distancia: 0.5301567912101746
ID: f8cb0b69-9f8a-492a-8e91-6c958e199fb5
Score: 0.5167
URL: https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg
Categoria: Mapa
Títulos: MAPA DE ÁGUAS SUBTERRÂNEAS DO ESTADO DE SÃO PAULO
Texts: #x
----------------------------------------
Distancia: 0.5167315006256104


In [81]:
def process_image_list(image_urls: list):

    for url in image_urls:
        print(f"\n[INFO] Processando imagem: {url}")
        process_and_store(url)


# 🔹 Exemplo de uso:
image_links = [
    "https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg",
    "https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEhFG7haT8p3MW18cI-97_LMRh9AWqdhI3S0NAVbLfiCH3h01KdutXPSo4mkAdQrm3TI1XyGhyphenhyphenFstf3bgaViSeMYQ6oKF4Q3yuUwLkwOuFLS-eRl6utPWFXxrQh5ytkkbx3STqyDs2CHRYhU/s2048/Biomas_SCM_2019_munic%25C3%25ADpios+2.png",
    "https://smastr16.blob.core.windows.net/2001/sites/261/2024/03/uso-do-solo-800x564.jpg",
    "https://biomalegal.com.br/wp-content/uploads/2018/06/mapa-areas-prioritarias-sp.jpg",
    "https://fflorestal.sp.gov.br/wp-content/uploads/2018/11/mapa-ucs-nov-1.png",
    "https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEhtNjvpU39nUZezymq2vtIm9FZ40Y40mId1DKKAaoIM1PTP4np-Qacw3R5Idn5kEHBhYGwLb5IUYJIIaWYDDc5UX6J1NM9hSw8WJPCdB2dkKZhTVyc0TSpCnV2qSx5TgLUy-tE4Au8SjkJ9/s2048/Regi%25C3%25B5es_fitoecol%25C3%25B3gicas.png",
    "https://www.researchgate.net/profile/R-Mingoti-2/publication/272177169/figure/fig1/AS:391940228042759@1470457240998/FIGURA-1-Hidrografia-do-Estado-de-Sao-Paulo-com-as-bacias-hidrograficas-das-estacoes.png",
    "https://smastr16.blob.core.windows.net/igeo/sites/233/2021/01/mapa_risco.jpg",
    "https://www.redalyc.org/journal/3211/321165166006/321165166006_gf2.png",
    "https://jornal.usp.br/wp-content/uploads/2025/01/20250123_mapa_sao-paulo_clima.jpg",
    "https://www.researchgate.net/publication/347929545/figure/fig1/AS:975279040245782@1609536051857/Figura-1-Mapa-hidrogeologico-do-estado-de-Sao-Paulo-A-Distribuicao-dos-pocos.ppm",
    "https://geoftp.ibge.gov.br/cartas_e_mapas/mapas_estaduais_e_distrito_federal/fisico/sp_fisico900k_2011.pdf",
    "https://g1.globo.com/Noticias/Ciencia/foto/0,,11974060,00.jpg",
    "https://infograficos.estadao.com.br/cidades/fauna-invisivel/img/infograficos-placeholder/habitalidadev2.png",
    "https://www.al.sp.gov.br/repositorio/legislacao/decreto/2018/decreto-63853-img1-27.11.2018.jpg?rnd=65381423",
    "https://smastr16.blob.core.windows.net/home/2020/07/2bbe2052-7138-42dd-a6ad-56c0c433b366-1024x723.jpg",
    "https://sigam.ambiente.sp.gov.br/sigam3/Repositorio/222/Documentos/mapas/A1_InvFlorestal.jpg",
    "https://storage.googleapis.com/spatialnodefiles/projects/6a37f831-db36-4e7e-a116-0579fbebd6d3Mapadenpop2021.jpeg",
    "https://www.nossosaopaulo.com.br/Reg_SP/Hist_Geog/HistGeo_ImagMap/SP_DDemog.jpg",
    "https://journals.openedition.org/confins/docannexe/image/7215/img-6.png",
    "https://www.researchgate.net/profile/Neli-Mello-Thery/publication/259481458/figure/fig1/AS:670721319653389@1536923832630/Figura-1-Mapa-de-uso-e-ocupacao-do-solo-da-Regiao-Metropolitana-de-Sao-Paulo.png",
    "https://www.researchgate.net/profile/Ruth-Ramos/publication/346971772/figure/fig6/AS:968356542828545@1607885599780/Figura-1-Mapa-de-uso-das-terras-no-estado-de-Sao-Paulo-em-2017-com-destaque-em-negrito.png",
    "https://www.researchgate.net/profile/Ruth-Ramos/publication/346971772/figure/fig6/AS:968356542828545@1607885599780/Figura-1-Mapa-de-uso-das-terras-no-estado-de-Sao-Paulo-em-2017-com-destaque-em-negrito.png",
    "https://upload.wikimedia.org/wikipedia/commons/d/df/Agricultura_no_Sudeste_do_Brasil.jpg",
    "https://ichef.bbci.co.uk/ace/ws/640/cpsprodpb/83f6/live/2f86e720-6f7d-11ef-8c1a-df523ba43a9a.jpg.webp",
    "https://energiaeambiente.org.br/wp-content/uploads/2023/05/mapa_IEMA_2023_1440x1080_final-1.jpg",
    "https://blog.coontrol.com.br/wp-content/uploads/2019/03/grafico-emissoes-totais-gases-poluentes.jpg.webp"
    "https://drive.prefeitura.sp.gov.br/cidade/secretarias/upload/x.png",
    "https://energiaeambiente.org.br/wp-content/uploads/2020/12/Figura-1-Emissoes-brutas-brasileiras-de-GEE-por-setor-1990-2019-1024x520.jpg",
    "https://lh6.googleusercontent.com/proxy/jijNwi8mCRsNJ3Df1vyNBP9J0o1HyoDBqvk48ezU-SmpzoOy12WNmLkqclExni01mcNHUHw_rP91YT1BuciretalQ_sO",
    "https://ufscsustentavel.paginas.ufsc.br/files/2018/11/Volume-Mensal-de-Água-na-UFSC-2017-e-2018.jpg",
    "https://www.gov.br/cetem/pt-br/assuntos/noticias/cetem-progride-na-reducao-do-consumo-anual-de-agua/consumo-anual-agua-v2.png",
    "https://jornal.usp.br/wp-content/uploads/2024/06/20240612_queda-populacao-sao-paulo.jpg",
    "https://projetocolabora.com.br/wp-content/uploads/2023/12/graficoeustaquio1.png",
    "https://projetocolabora.com.br/wp-content/uploads/2022/12/eustaquio-38.png",
    "https://investsp.org.br/wp-content/uploads/2025/03/sao_paulo-2011.jpg",
    "https://canalsolar.com.br/wp-content/uploads/2023/03/unnamed.png"
]

process_image_list(image_links)



[INFO] Processando imagem: https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg
[SKIP] Imagem já existe no Milvus (ID f8cb0b69-9f8a-492a-8e91-6c958e199fb5)

[INFO] Processando imagem: https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEhFG7haT8p3MW18cI-97_LMRh9AWqdhI3S0NAVbLfiCH3h01KdutXPSo4mkAdQrm3TI1XyGhyphenhyphenFstf3bgaViSeMYQ6oKF4Q3yuUwLkwOuFLS-eRl6utPWFXxrQh5ytkkbx3STqyDs2CHRYhU/s2048/Biomas_SCM_2019_munic%25C3%25ADpios+2.png


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID 270dadf2-a235-4994-b7de-7163c5276b11

[INFO] Processando imagem: https://smastr16.blob.core.windows.net/2001/sites/261/2024/03/uso-do-solo-800x564.jpg


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID 6ab5454f-76c3-4385-9938-3f36c25bcce8

[INFO] Processando imagem: https://biomalegal.com.br/wp-content/uploads/2018/06/mapa-areas-prioritarias-sp.jpg


...

[OK] Imagem armazenada no Milvus com ID 9a7e6151-9501-48e7-a212-c2b2ec8968ca

[INFO] Processando imagem: https://projetocolabora.com.br/wp-content/uploads/2022/12/eustaquio-38.png


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID def6b9f1-906c-40bf-b407-8ce5074d9ad1

[INFO] Processando imagem: https://investsp.org.br/wp-content/uploads/2025/03/sao_paulo-2011.jpg
Erro ao processar/armazenar imagem: cannot identify image file <_io.BytesIO object at 0x000001BF9D426F20>

[INFO] Processando imagem: https://canalsolar.com.br/wp-content/uploads/2023/03/unnamed.png


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID 912c0114-f747-4b0a-bfcd-25b686b1a4aa


In [82]:
def get_all_images():

    try:
        # Consulta todos os registros (limit pode ser ajustado)
        results = collection.query(
            expr="",  # sem filtro, retorna todos
            output_fields=["id", "url", "category", "titles", "texts"],
            limit=1000  # limite de segurança, pode aumentar conforme necessário
        )

        imagens = []
        for r in results:
            descricao = f"{r['category']}: {r['titles']} {r['texts']}".strip()
            imagens.append({
                "id": r["id"],
                "url": r["url"],
                "category": r["category"],
                "titles": r["titles"],
                "texts": r["texts"],
                "descricao": descricao
            })

        return imagens

    except Exception as e:
        print(f"Erro ao recuperar imagens: {e}")
        return []


# 🔹 Exemplo de uso:
todas = get_all_images()
for img in todas:
    print(f"\n🖼️ {img['url']}")
    print(f"📌 Categoria: {img['category']}")
    print(f"📖 Título: {img['titles']}")
    print(f"📝 Texto: {img['texts']}")
    print(f"🧾 Descrição: {img['descricao']}")



🖼️ https://www.redalyc.org/journal/3211/321165166006/321165166006_gf2.png
📌 Categoria: Mapa
📖 Título: 
📝 Texto: 200
🧾 Descrição: Mapa:  200

🖼️ https://jornal.usp.br/wp-content/uploads/2025/01/20250123_mapa_sao-paulo_clima.jpg
📌 Categoria: Mapa
📖 Título: 50' W 48ª W N
📝 Texto: 8 Mato Grosso do Sul Minas Gerais Rio Annual Precipitation 1100 mm 1200 mm 1300 mm 1400 mm 1500 mm 2000 mm 2500 mm Paraná 100 50 00 3000 mm Km
🧾 Descrição: Mapa: 50' W 48ª W N 8 Mato Grosso do Sul Minas Gerais Rio Annual Precipitation 1100 mm 1200 mm 1300 mm 1400 mm 1500 mm 2000 mm 2500 mm Paraná 100 50 00 3000 mm Km

🖼️ https://ufscsustentavel.paginas.ufsc.br/files/2018/11/Volume-Mensal-de-Água-na-UFSC-2017-e-2018.jpg
📌 Categoria: Gráfico
📖 Título: Volume Mensal de Água da UFSC 2017 2018 30.000
📝 Texto: 5.000 2017 Z015 Média 2017
🧾 Descrição: Gráfico: Volume Mensal de Água da UFSC 2017 2018 30.000 5.000 2017 Z015 Média 2017

🖼️ https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEhFG7haT8p3MW18cI-97_LMRh9

In [69]:
# from pymilvus import connections, utility

# # Garante que está conectado (desconectando antes, só por segurança)
# connections.disconnect("default")
# connections.connect("default", host="localhost", port="19530")

# # Nome da coleção
# collection_name = "image_descriptions"

# if utility.has_collection(collection_name):
#     utility.drop_collection(collection_name)
#     print(f"Coleção '{collection_name}' excluída com sucesso.")
# else:
#     print(f"Coleção '{collection_name}' não existe.")


Coleção 'image_descriptions' excluída com sucesso.
